# 📈 Learning Rate Schedules: Cosine vs. WSD (Warmup-Stable-Decay)

Welcome! In this notebook, we explore <b>Learning Rate Schedulers</b>—the throttle control of foundation model pretraining.

---

## 1. The Metaphor: The Spacecraft Landing 🚀

Imagine piloting a spacecraft landing on a new planet:
<ol>
  <li><b>Warmup Phase (Takeoff):</b> Gently ramp up throttle from 0% to 100% so the ship doesn't shake apart from cold engines (prevents gradient explosion).</li>
  <li><b>Stable Phase (Cruising):</b> Cruise at maximum constant speed across the vast distance to learn general representations efficiently.</li>
  <li><b>Decay Phase (Landing):</b> As you approach the surface, throttle down by 90% to make a pinpoint landing into the exact optimal valley.</li>
</ol>

---
## 2. Cosine vs. WSD Architecture Comparison

<pre style='font-size: 13px; background: #0d1117; color: #c9d1d9; padding: 16px; border-radius: 8px; border: 1px solid #30363d;'>
1. Classic Cosine Decay (GPT-3, LLaMA-1/2):
   LR
    ▲   / \
    │  /   ` - .
    │ /          ` - . _
    │/                   ` - . _
    └───────────────────────────► Steps
      Warmup       Decays continuously across the entire run

   ❌ The Flaw: You MUST set the exact step count T before starting!
      If you want to train longer, you cannot cleanly extend it.

2. Modern WSD (Warmup-Stable-Decay) (MiniCPM, Mirendil-Base):
   LR
    ▲   /──────────────\
    │  /  STABLE PHASE  \
    │ /   (Constant LR)  \  DECAY
    │/                    \
    └──────────────────────► Steps
     Warmup   Indefinite!   Final 15%

   ✅ The Breakthrough: You can train in the Stable Phase for as long as you want!
      Whenever you decide to finish the run, trigger Decay for the final 15% to land!
</pre>

---
## 3. Exercise 1: Pencil & Paper Calculation 📝

Suppose our training run has:
- Max Learning Rate = <b>0.001</b>
- Total Steps = <b>100</b>
- Warmup = Steps 0 to 20
- Stable = Steps 20 to 80
- Decay = Steps 80 to 100 (anneals down to 0.0001)

### Calculate the LR at 4 Key Points on Paper:
1. <b>Step 10 (Halfway through warmup):</b><br>
   LR = 0.001 · (10 / 20) = <b>0.0005</b>
<br><br>
2. <b>Step 50 (Middle of Stable phase):</b><br>
   LR = <b>0.0010</b> (Full throttle!)
<br><br>
3. <b>Step 80 (End of Stable phase):</b><br>
   LR = <b>0.0010</b> (Still full throttle!)
<br><br>
4. <b>Step 100 (Final landing step):</b><br>
   LR = <b>0.0001</b> (90% decay &rarr; settles into sharp global minimum!)

<br>
<i>Notice: During steps 80 to 100, validation loss plunges like a waterfall!</i>

---
## 4. Visual Comparison: Cosine vs. WSD in Matplotlib

Run this cell to plot our actual project schedulers side-by-side:

In [ ]:
import sys
from pathlib import Path
root_dir = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(root_dir) not in sys.path:
    sys.path.insert(0, str(root_dir))

import torch
import matplotlib.pyplot as plt
from src.engine.optim import get_lr_scheduler

total_steps = 1000
warmup_steps = 100

# Mock optimizer
p = torch.nn.Parameter(torch.zeros(1))
opt_cosine = torch.optim.SGD([p], lr=1e-3)
opt_wsd = torch.optim.SGD([p], lr=1e-3)

sched_cosine = get_lr_scheduler(opt_cosine, scheduler_type='cosine', warmup_steps=warmup_steps, total_steps=total_steps)
sched_wsd = get_lr_scheduler(opt_wsd, scheduler_type='wsd', warmup_steps=warmup_steps, total_steps=total_steps, stable_ratio=0.8)

lrs_cosine = []
lrs_wsd = []

for step in range(total_steps):
    lrs_cosine.append(sched_cosine.get_last_lr()[0])
    lrs_wsd.append(sched_wsd.get_last_lr()[0])
    sched_cosine.step()
    sched_wsd.step()

fig, ax = plt.subplots(figsize=(9, 4.5))
ax.plot(lrs_cosine, label='Cosine Schedule (Decays continuously)', color='#d29922', lw=2)
ax.plot(lrs_wsd, label='WSD Schedule (Stable plateau then decay)', color='#58a6ff', lw=2.5)

ax.set_title("Learning Rate Schedule: Cosine vs. WSD (Warmup-Stable-Decay)", fontsize=12, fontweight='bold', pad=12)
ax.set_xlabel("Training Steps", fontsize=10)
ax.set_ylabel("Learning Rate", fontsize=10)
ax.grid(True, alpha=0.3)
ax.legend(loc='upper right')

plt.tight_layout()
plt.show()
print("SUCCESS: Schedulers traced and plotted cleanly!")